# ESMC Biohub - Remote Inference via Biohub Platform API

Run ESMC remotely via Biohub Platform using Forge credentials for authentication.

**Key Features:**
- Remote inference via Biohub API (no GPU needed locally)
- Authenticate with Forge credentials
- Batch processing of sequences
- Extract embeddings and logits from ESMC
- Compute Shannon entropy and mutation scores


In [ ]:
!pip install -q requests pandas numpy tqdm ipywidgets scipy scikit-learn

In [ ]:
# Install ESM from GitHub
!pip install -q git+https://github.com/Biohub/esm.git@main

In [ ]:
import requests
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional
from tqdm import tqdm
from IPython.display import display, Markdown
import base64
from scipy.special import softmax

print("Dependencies loaded successfully")

## Authentication & Configuration

In [ ]:
#@markdown Enter your Biohub Forge credentials

from getpass import getpass

# Get credentials
FORGE_USERNAME = input("Forge Username: ")
FORGE_API_KEY = getpass("Forge API Key: ")

# Biohub API endpoint
BIOHUB_API_URL = "https://api.biohub.org/v1"
MODEL_NAME = "esmc_600m"  # Options: esmc_600m, esmc_300m, esmc_300m_open_v1
OUTPUT_DIR = "/content/esmc_biohub_output"
Path(OUTPUT_DIR).mkdir(exist_ok=True)

# Create auth header
auth_string = base64.b64encode(f"{FORGE_USERNAME}:{FORGE_API_KEY}".encode()).decode()
HEADERS = {
    "Authorization": f"Basic {auth_string}",
    "Content-Type": "application/json"
}

print(f"✓ Credentials configured")
print(f"✓ Output directory: {OUTPUT_DIR}")

## Utility Functions

In [ ]:
def load_fasta(filepath: str) -> Dict[str, str]:
    """Load FASTA file into dictionary."""
    sequences = {}
    with open(filepath) as f:
        current_id = None
        current_seq = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    sequences[current_id] = "".join(current_seq)
                current_id = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
        if current_id is not None:
            sequences[current_id] = "".join(current_seq)
    return sequences

def call_esmc_api(sequence: str, task: str = "embed") -> Optional[Dict]:
    """Call Biohub ESMC API.
    
    Args:
        sequence: Protein sequence
        task: "embed" (get embeddings) or "logits" (get logits)
    """
    payload = {
        "model": MODEL_NAME,
        "sequence": sequence,
        "task": task
    }
    
    try:
        response = requests.post(
            f"{BIOHUB_API_URL}/esmc/inference",
            headers=HEADERS,
            json=payload,
            timeout=30
        )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"API error for sequence: {e}")
        return None

def compute_entropy(logits: np.ndarray) -> np.ndarray:
    """Compute Shannon entropy from logits."""
    probs = softmax(logits, axis=-1)
    entropy = -np.sum(probs * np.log(probs + 1e-10), axis=-1)
    return entropy

## Load Sequences

In [ ]:
#@markdown Upload FASTA file or use example sequences

from google.colab import files

input_mode = "paste"  # "upload" or "paste"

if input_mode == "upload":
    print("Upload your FASTA file:")
    uploaded = files.upload()
    fasta_file = list(uploaded.keys())[0]
    sequences = load_fasta(fasta_file)
else:
    # Example sequences
    sequences = {
        "GFP": "MVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVAWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK",
        "MBP": "MKIEEGKLPIBQFEVDWSGYSGTWFTAGQTFTVTE"
    }

print(f"Loaded {len(sequences)} sequences")
for seq_id, seq in list(sequences.items())[:3]:
    print(f"  {seq_id}: {len(seq)} aa")

## Run ESMC Inference

In [ ]:
print(f"Processing {len(sequences)} sequences with ESMC ({MODEL_NAME})...\n")

results = {}
failed = []

for seq_id, sequence in tqdm(sequences.items(), desc="Inference"):
    # Get embeddings
    result = call_esmc_api(sequence, task="embed")
    
    if result is None:
        failed.append(seq_id)
        continue
    
    # Get logits for entropy computation
    logits_result = call_esmc_api(sequence, task="logits")
    
    if logits_result is None:
        logits = None
    else:
        logits = np.array(logits_result.get("logits", []))
    
    results[seq_id] = {
        "embedding": np.array(result.get("embedding", [])),
        "logits": logits,
        "length": len(sequence),
        "sequence": sequence
    }

print(f"\n✓ Processed {len(results)} sequences successfully")
if failed:
    print(f"✗ Failed: {', '.join(failed)}")

## Entropy Analysis

In [ ]:
import matplotlib.pyplot as plt

entropy_data = {}

for seq_id, data in results.items():
    if data["logits"] is not None:
        entropy = compute_entropy(data["logits"])
        entropy_data[seq_id] = entropy

if entropy_data:
    fig, axes = plt.subplots(len(entropy_data), 1, figsize=(14, 3*len(entropy_data)))
    if len(entropy_data) == 1:
        axes = [axes]
    
    for idx, (seq_id, entropy) in enumerate(entropy_data.items()):
        ax = axes[idx]
        ax.plot(entropy, linewidth=2, color='steelblue')
        ax.fill_between(range(len(entropy)), entropy, alpha=0.3, color='steelblue')
        ax.set_title(f"{seq_id} - Shannon Entropy (Conservation Analysis)")
        ax.set_xlabel("Position")
        ax.set_ylabel("Entropy (bits)")
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/entropy_analysis.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    display(Markdown("**Interpretation:**\n- High entropy → flexible/variable positions (mutation-tolerant)\n- Low entropy → conserved/constrained positions (function-critical)"))
else:
    print("No logits available for entropy computation")

## Summary Statistics

In [ ]:
stats = []

for seq_id, data in results.items():
    embedding = data["embedding"]
    
    stat = {
        "sequence_id": seq_id,
        "length": data["length"],
        "embedding_dim": embedding.shape[1] if len(embedding.shape) > 1 else 0,
    }
    
    if seq_id in entropy_data:
        entropy = entropy_data[seq_id]
        stat.update({
            "mean_entropy": float(np.mean(entropy)),
            "min_entropy": float(np.min(entropy)),
            "max_entropy": float(np.max(entropy)),
        })
    
    stats.append(stat)

df_stats = pd.DataFrame(stats)
display(df_stats)

# Save statistics
df_stats.to_csv(f"{OUTPUT_DIR}/summary_statistics.csv", index=False)
print(f"\n✓ Statistics saved to {OUTPUT_DIR}/summary_statistics.csv")

## Save Results

In [ ]:
# Save embeddings and entropy
for seq_id, data in results.items():
    # Embeddings
    np.save(f"{OUTPUT_DIR}/{seq_id}_embedding.npy", data["embedding"])
    
    # Entropy
    if seq_id in entropy_data:
        entropy_df = pd.DataFrame({
            "position": range(len(entropy_data[seq_id])),
            "entropy": entropy_data[seq_id]
        })
        entropy_df.to_csv(f"{OUTPUT_DIR}/{seq_id}_entropy.csv", index=False)

print(f"✓ All results saved to {OUTPUT_DIR}/")
print(f"  - Embeddings: {{seq_id}}_embedding.npy")
print(f"  - Entropy: {{seq_id}}_entropy.csv")
print(f"  - Statistics: summary_statistics.csv")